# 💻 Chapter 19: Randomized Algorithms
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** Sometimes the best algorithm for a problem is a coin flip. Randomized quicksort beats deterministic worst-case. QuickSelect finds medians in O(n). And hash functions are probabilistic magic.

**🎯 Objectives:** Understand randomized quicksort, reservoir sampling, hashing, and why randomness helps algorithms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# ── Randomized QuickSort ──
def quicksort_deterministic(arr):
    if len(arr) <= 1: return arr
    pivot = arr[0]  # always first element (bad on sorted input!)
    left  = [x for x in arr[1:] if x <= pivot]
    right = [x for x in arr[1:] if x > pivot]
    return quicksort_deterministic(left) + [pivot] + quicksort_deterministic(right)

def quicksort_random(arr, rng=None):
    if rng is None: rng = np.random.default_rng()
    if len(arr) <= 1: return arr
    pivot_idx = rng.integers(len(arr))  # RANDOM pivot
    pivot = arr[pivot_idx]
    left  = [x for x in arr if x < pivot]
    mid   = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort_random(left, rng) + mid + quicksort_random(right, rng)

import sys
sys.setrecursionlimit(5000)

# Compare on sorted input (worst case for deterministic)
n = 200
sorted_arr = list(range(n))

t0 = time.perf_counter()
result_det = quicksort_deterministic(sorted_arr.copy())
t_det = time.perf_counter() - t0

t0 = time.perf_counter()
result_rand = quicksort_random(sorted_arr.copy(), rng)
t_rand = time.perf_counter() - t0

print("⚡ QuickSort: Sorted Input (Worst Case for Deterministic)")
print(f"  Deterministic: {t_det*1000:.2f}ms (O(n²) on sorted input!)")
print(f"  Randomized:    {t_rand*1000:.2f}ms (O(n log n) expected)")
print()

# Distribution of pivot positions (why random pivot is good)
n_trials = 10_000
arr = list(range(100))
pivot_positions = [rng.integers(100) for _ in range(n_trials)]
# Fraction in each "quartile"
q25 = sum(1 for p in pivot_positions if 25 <= p <= 75) / n_trials
print(f"P(pivot in middle 50%) = {q25:.3f}")
print("Good pivot → balanced split → O(n log n) expected")

In [ ]:
# ── Skip List: Probabilistic Data Structure ──
# Show that random coin flips create balanced layers
import bisect

class SkipList:
    def __init__(self, max_level=4, p=0.5):
        self.max_level = max_level
        self.p = p
        self.rng = np.random.default_rng(42)
        self.layers = [[] for _ in range(max_level)]

    def _random_level(self):
        level = 1
        while self.rng.random() < self.p and level < self.max_level:
            level += 1
        return level

    def insert(self, val):
        level = self._random_level()
        for l in range(level):
            bisect.insort(self.layers[l], val)

    def search(self, val, verbose=False):
        for l in range(self.max_level-1, -1, -1):
            if val in self.layers[l]:
                if verbose: print(f"  Found in layer {l}!")
                return True
        return False

sl = SkipList()
for v in rng.integers(1, 100, 50):
    sl.insert(int(v))

print("Skip List Layer Sizes:")
for i, layer in enumerate(sl.layers):
    print(f"  Layer {i}: {len(layer)} elements {'▓'*len(layer)}")

print(f"
Search for 42:")
sl.search(42, verbose=True)

# Hash function collision probability
def birthday_hash_collision(n_items, table_size):
    # Approximate P(at least one collision) for n items in table of size m.
    return 1 - np.exp(-n_items**2 / (2*table_size))

print("
🔑 Hash Table Collision Probability:")
table_sizes = [100, 1000, 10000]
for m in table_sizes:
    print(f"
  Table size m={m}:")
    for n in [10, int(m**0.5), m//2]:
        p_coll = birthday_hash_collision(n, m)
        print(f"    n={n:>6} items: P(collision) ≈ {p_coll:.3f}")

In [ ]:
# ── Reservoir Sampling: Uniform sample from unknown-length stream ──
def reservoir_sample(stream_generator, k, seed=42):
    rng = np.random.default_rng(seed)
    reservoir = []
    for i, item in enumerate(stream_generator):
        if i < k:
            reservoir.append(item)
        else:
            j = rng.integers(0, i+1)
            if j < k:
                reservoir[j] = item
    return reservoir

# Verify uniformity: each element should have equal P(in reservoir)
N = 10_000  # stream length
k = 100    # reservoir size
n_trials = 1000

# Track how often each position in stream appears in reservoir
appearance_count = np.zeros(N)
for _ in range(n_trials):
    reservoir = reservoir_sample(iter(range(N)), k, seed=np.random.randint(0, 10000))
    for item in reservoir:
        appearance_count[item] += 1

empirical_probs = appearance_count / n_trials
theoretical_prob = k / N

print(f"Reservoir Sampling Uniformity Test")
print(f"  k={k}, N={N}, n_trials={n_trials}")
print(f"  Theoretical P(each item) = {theoretical_prob:.4f}")
print(f"  Empirical P (mean):        {empirical_probs.mean():.4f}")
print(f"  Empirical P (std):         {empirical_probs.std():.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(empirical_probs, alpha=0.5, color='#3498db', label='Empirical P')
ax.axhline(theoretical_prob, color='red', lw=2, linestyle='--', label=f'Theoretical P={theoretical_prob}')
ax.set_xlabel('Stream Position'); ax.set_ylabel('P(in reservoir)')
ax.set_title(f'Reservoir Sampling: Every position equally likely ✅', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('ch19_reservoir.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Implement QuickSelect (find kth smallest in O(n) expected time) using random pivot.

**Challenge 2:** Generate a random permutation in O(n) using Fisher-Yates shuffle.
Verify that all permutations of [1,2,3] are equally likely.

**Challenge 3:** Implement a simple Bloom filter and measure the empirical false positive rate.

<details><summary>Solutions</summary>See code below.</details>

In [ ]:
# Fisher-Yates shuffle
def fisher_yates(arr, seed=None):
    arr = arr.copy()
    rng = np.random.default_rng(seed)
    for i in range(len(arr)-1, 0, -1):
        j = rng.integers(0, i+1)
        arr[i], arr[j] = arr[j], arr[i]
    return arr

# Verify uniformity for [1,2,3]
from collections import Counter
from itertools import permutations

results = Counter()
for _ in range(60000):
    perm = tuple(fisher_yates([1,2,3]))
    results[perm] += 1

all_perms = list(permutations([1,2,3]))
print("Fisher-Yates Permutation Uniformity:")
for perm in all_perms:
    count = results[perm]
    print(f"  {perm}: {count} times ({count/600:.1f}%) — expected 100%")

# Simple Bloom filter
import hashlib
class BloomFilter:
    def __init__(self, size=1000, n_hash=3):
        self.bits = np.zeros(size, dtype=bool)
        self.size = size
        self.n_hash = n_hash

    def _hashes(self, item):
        return [int(hashlib.md5(f"{item}{i}".encode()).hexdigest(), 16) % self.size
                for i in range(self.n_hash)]

    def add(self, item):
        for h in self._hashes(item): self.bits[h] = True

    def __contains__(self, item):
        return all(self.bits[h] for h in self._hashes(item))

bf = BloomFilter(size=5000, n_hash=3)
for i in range(500): bf.add(f"user_{i}")

# False positives
fps = sum(1 for i in range(500, 2000) if f"user_{i}" in bf)
print(f"
Bloom Filter: 500 elements, 1500 queries for unknown items")
print(f"False positives: {fps}/1500 = {fps/1500:.2%}")

## 🎯 Recap
1. Random pivot in QuickSort avoids worst-case O(n²) — expected O(n log n).
2. Reservoir sampling gives uniform samples from streams with unknown length.
3. Hash functions are probabilistic — birthday problem governs collision probability.

**Next:** [Chapter 20 — Bloom Filters & Probabilistic Data Structures]